#### Sentiment in NLP

In NLP (Natural Language Processing), sentiment refers to the emotional tone or attitude expressed in text — whether it is positive, negative, or neutral. It’s essentially about detecting how people feel from their words.

- Positive sentiment → words like happy, excellent, love.
- Negative sentiment → words like bad, terrible, hate.
- Neutral sentiment → words like movie, table, was (no emotional charge).

#### Rule‑Based Sentiment Analysis in NLP:

It’s a non‑ML approach that uses **predefined lexicons (word lists)** and rules to determine sentiment. Each word has an associated polarity score (positive, negative, neutral). The overall sentiment of a sentence is computed by aggregating scores according to rules.

Top Pre-built Lexicons and Libraries:

- **VADER** (Valence Aware Dictionary and sEntiment Reasoner): Specifically designed for social media text, VADER handles emojis, slang (e.g., "lol"), and capitalizations. It is available as a standalone package via PyPI or as part of the NLTK library.

- **TextBlob**: This library includes a pre-trained lexicon (based on the Pattern library) that provides two main scores: Polarity (-1.0 to 1.0) and Subjectivity (0.0 to 1.0). It is known for being easy to use for beginners.

- **AFINN**: One of the simplest and most popular lexicons, AFINN contains over 2,400 English words rated for valence with an integer between -5 and +5.
SentiWordNet: An extension of WordNet where each "synset" (group of synonyms) is associated with three numerical scores: positivity, negativity, and objectivity. It is accessible through NLTK.

- Loughran-McDonald: A specialized lexicon for financial sentiment analysis, often used to analyze earnings reports and 10-K filings, as it accounts for financial jargon that might be misclassified by general lexicons.

#### Qus: How Lexicon Scores Are Assigned?

Lexicon scores come from **pre‑built sentiment dictionaries** created by linguists or data scientists. Each word is given a polarity value based on human annotation, statistical frequency, or corpus analysis.

1. Human Annotation
    - Experts manually label words with sentiment strength.
    - Example: "excellent" → +0.9, "terrible" → −0.9.
    - Neutral words like "movie" or "was" → 0.

2. Corpus Statistics

* Words co-occurring with positive/negative contexts in large datasets are scored.
* Formula (Pointwise Mutual Information, PMI):

$$score(w) = \log \frac{P(w, positive)}{P(w) \cdot P(positive)} - \log \frac{P(w, negative)}{P(w) \cdot P(negative)}$$

* If $score(w) > 0 \rightarrow$ positive tendency.
* If $score(w) < 0 \rightarrow$ negative tendency.
* If $score(w) \approx 0 \rightarrow$ neutral.

3. Normalization
Scores are scaled into $[-1, 1]$ range:

$$score_{norm}(w) = \frac{score(w)}{\max(|score(w)|)}$$

4. Example Calculation

Suppose corpus statistics:

* $P(good, positive) = 0.05$
* $P(good) = 0.06$
* $P(positive) = 0.5$
* $P(good, negative) = 0.01$
* $P(negative) = 0.5$

$$
\begin{aligned}
score(good) &= \log \frac{0.05}{0.06 \cdot 0.5} - \log \frac{0.01}{0.06 \cdot 0.5} \\
\\
&= \log \frac{0.05}{0.03} - \log \frac{0.01}{0.03} \\
\\
&= \log(1.67) - \log(0.33) \approx 0.51 - (-1.10) = 1.61
\end{aligned}
$$

Normalize to $[-1, 1]$:

$$score_{norm}(good) \approx +0.7$$

That’s how "good" gets a positive score, while "movie" (neutral) stays at 0.

#### Lexicon Scores in Sentiment Analysis

A **sentiment lexicon** is a dictionary where each word is assigned a **polarity score**:

$$score(w) \in [-1, 1]$$

* $+1 \rightarrow$ strongly positive
* $-1 \rightarrow$ strongly negative
* $0 \rightarrow$ neutral

Sentence Sentiment Formula

##### 1. Raw Sentiment:

$$Sentiment(S) = \sum_{i=1}^{n} score(w_i)$$

##### 2. Normalized Sentiment:

$$Sentiment_{norm}(S) = \frac{Sentiment(S)}{n}$$

##### 3. Rule Adjustments:

* **Negation:** If "not" precedes a positive word, flip sign:
$$score(w) \rightarrow -score(w)$$

* **Intensity Modifiers:** If "very" precedes a word, amplify:
$$score(w) \rightarrow \alpha \cdot score(w), \quad \alpha > 1$$

* **Punctuation/Emphasis:** Exclamation marks can boost:
$$score(S) \rightarrow score(S) + \beta$$

Example Walkthrough

Sentence:

"The movie was very good but not perfect!"

### Step 1: Lexicon Scores

*   `movie` $\rightarrow 0$
*   `good` $\rightarrow +0.7$
*   `very` $\rightarrow$ intensity modifier ($\alpha = 1.5$)
*   `not` $\rightarrow$ negation rule
*   `perfect` $\rightarrow +0.9$
*   `but` $\rightarrow 0$
*   `was` $\rightarrow 0$

### Step 2: Apply Rules

*   "very good" $\rightarrow 1.5 \times 0.7 = +1.05$
*   "not perfect" $\rightarrow$ flip polarity $\rightarrow -0.9$

---

### Step 3: Aggregate

$$Sentiment(S) = 1.05 + (-0.9) = +0.15$$

### Step 4: Normalize

- Sentence length $n = 7$:
- $$Sentiment_{norm}(S) = \frac{0.15}{7} \approx +0.021$$

### Step 5: Punctuation Boost

- Exclamation mark ($\beta = 0.1$):
- $$Final\_Sentiment = 0.021 + 0.1 = +0.121$$

👉 **Interpretation:** Slightly positive sentiment.




##### Sentiment analysis pipeline on the IMDb dataset using VADER:



In [1]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


In [2]:
# Assuming dataset is in CSV format
df = pd.read_csv("./IMDB Dataset.csv")
print(df.head())

analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    return analyzer.polarity_scores(text)

df['scores'] = df['review'].apply(vader_sentiment)
df['compound'] = df['scores'].apply(lambda x: x['compound'])


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


#### Convert Compound Score to Labels

- compound ≥ 0.05 → Positive
- compound ≤ −0.05 → Negative
- otherwise → Neutral

In [3]:
df['vader_label'] = df['compound'].apply(
    lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
)

In [4]:
from sklearn.metrics import accuracy_score, classification_report

# Evaluate Against IMDb Labels
print("Accuracy:", accuracy_score(df['sentiment'], df['vader_label']))
print(classification_report(df['sentiment'], df['vader_label']))


Accuracy: 0.69226


c:\Users\TODO\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\TODO\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

    negative       0.79      0.53      0.63     25000
     neutral       0.00      0.00      0.00         0
    positive       0.65      0.86      0.74     25000

    accuracy                           0.69     50000
   macro avg       0.48      0.46      0.46     50000
weighted avg       0.72      0.69      0.69     50000



c:\Users\TODO\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
